# Run prompt-conditioned MIDI inference

Set the four parameters below, then run all cells. The model operates on
two-bar patterns, so this notebook modifies the first two-bar window of the
input MIDI and writes one output MIDI file.


In [ ]:
# Inference parameters
MODEL_PATH = "text2groove_model.pt"
INPUT_MIDI_PATH = "Example Input MIDI/pattern_19_102bpm.mid"
PROMPT = "Humanize this as rock with tight, precise timing, use strong dynamics."
OUTPUT_MIDI_PATH = "modified_output.mid"

# Other prompts to try 
# PROMPT = "Make this sound like rock with swing, keep the intensity high."
# PROMPT = "Give this a funk groove with a relaxed pocket, play it with medium strength."
# PROMPT = "Make this sound like jazz with swing, use gentle dynamics."

# Set to a number to override the MIDI file's initial tempo.
TEMPO_OVERRIDE = None

# Change the seed to sample a different valid humanization.
RANDOM_SEED = 123


In [369]:
import h5py
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import pretty_midi
import random

from sentence_transformers import SentenceTransformer
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [370]:
# Unique styles in the dataset:
# afrobeat
# afrocuban
# afrocuban/bembe
# afrocuban/calypso
# afrocuban/rhumba
# blues/shuffle
# country
# dance/breakbeat
# dance/disco
# funk
# funk/fast
# funk/latin
# funk/purdieshuffle
# funk/rock
# gospel
# highlife
# hiphop
# jazz
# jazz/fast
# jazz/funk
# jazz/fusion
# jazz/klezmer
# jazz/linear
# jazz/march
# jazz/mediumfast
# jazz/swing
# latin
# latin/bomba
# latin/bounce
# latin/brazilian
# latin/brazilian-baiao
# latin/brazilian-bossa
# latin/brazilian-frevo
# latin/brazilian-ijexa
# latin/brazilian-maracatu
# latin/brazilian-samba
# latin/brazilian-sambareggae
# latin/brazilian-songo
# latin/chacarera
# latin/dominican-merengue
# latin/merengue
# latin/reggaeton
# latin/samba
# latin/venezuelan-joropo
# latin/venezuelan-sangueo
# middleeastern
# neworleans/chacha
# neworleans/funk
# neworleans/secondline
# neworleans/shuffle
# pop
# pop/soft
# punk
# reggae
# reggae/slow
# rock
# rock/folk
# rock/halftime
# rock/indie
# rock/prog
# rock/rockabilly
# rock/shuffle
# soul
# soul/motown

GENRES = [
    "funk",
    "rock",
    "jazz",
    "hiphop",
]

FEEL_PHRASES = {
    "swung": [
        "with a swung feel",
        "with swing",
        "with delayed offbeats",
    ],
    "tight": [
        "with tight, precise timing",
        "locked to the grid",
        "with a straight pocket",
    ],
    "loose": [
        "with a relaxed pocket",
        "with laid-back timing",
        "with loose timing variation",
    ],
}

DYNAMICS_PHRASES = {
    "soft": [
        "play it softly",
        "use gentle dynamics",
        "keep the intensity low",
    ],
    "medium": [
        "use moderate dynamics",
        "keep the intensity balanced",
        "play with medium strength",
    ],
    "loud": [
        "play it loud",
        "use strong dynamics",
        "keep the intensity high",
    ],
}

PROMPT_PATTERNS = [
    "Give this a {genre} groove {feel}, {dynamics}.",
    "Humanize this as {genre} {feel}, {dynamics}.",
    "Make this sound like {genre} {feel}, {dynamics}.",
]


def _decode_h5_strings(values):
    # Decode a hdf5 string dataset to a np string array
    decoded = []
    for value in np.asarray(values):
        if isinstance(value, (bytes, np.bytes_)):
            decoded.append(value.decode("utf-8"))
        else:
            decoded.append(str(value))
    return np.asarray(decoded, dtype=str)


def create_attribute_prompt(genre, feel, overall_dynamics, random_prompt=True):
    # Build a prompt from the metadata attributes for one groove.
    # Validation/test use the first variant of each combination so repeated runs are deterministic.
    
    genre = str(genre).strip().lower() or "drum"
    feel = str(feel).strip().lower()
    overall_dynamics = str(overall_dynamics).strip().lower()

    chooser = random.choice if random_prompt else lambda values: values[0]

    return chooser(PROMPT_PATTERNS).format(
        genre=genre,
        feel=chooser(FEEL_PHRASES[feel]),
        dynamics=chooser(DYNAMICS_PHRASES[overall_dynamics]),
    )


In [371]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEXT_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"

text_encoder = SentenceTransformer(TEXT_ENCODER_NAME)
text_encoder.to(device)
text_encoder.eval()

# Freeze all text encoder parameters
for param in text_encoder.parameters():
    param.requires_grad = False

TEXT_EMB_DIM = text_encoder.get_embedding_dimension()

print("Device:", device)
print("Text embedding dim:", TEXT_EMB_DIM)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5783.31it/s]


Device: cpu
Text embedding dim: 384


In [372]:
# Encode the prompt strings produced by create_attribute_prompt.
# The cache avoids rerunning the frozen text encoder for wording variants
# that have already appeared in an earlier batch.
PROMPT_EMBEDDING_CACHE = {}


@torch.no_grad()
def encode_prompt_batch(prompts):
    if isinstance(prompts, str):
        prompts = [prompts]
    else:
        prompts = [str(prompt) for prompt in prompts]

    missing_prompts = list(dict.fromkeys(
        prompt
        for prompt in prompts
        if prompt not in PROMPT_EMBEDDING_CACHE
    ))

    if missing_prompts:
        missing_embeddings = text_encoder.encode(missing_prompts, batch_size=min(64, len(missing_prompts)),
            convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

        for prompt, embedding in zip(missing_prompts, missing_embeddings):
            PROMPT_EMBEDDING_CACHE[prompt] = torch.tensor(embedding, dtype=torch.float32)

    return torch.stack([PROMPT_EMBEDDING_CACHE[prompt] for prompt in prompts], dim=0).to(device)


In [373]:
# Text-conditioned conditional groove VAE

# transform the Sentence Transformer embedding into the size expected by the groove model
class TextConditionProjection(nn.Module):
    def __init__(self, text_emb_dim, d_model):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(text_emb_dim),
            nn.Linear(text_emb_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
        )

    def forward(self, text_emb):
        return self.net(text_emb)

class ConditionalGrooveVAE(nn.Module):
    """
    Training posterior:
        q(z | hits, target offsets, target velocities, text, tempo)

    Decoder:
        p(offsets, velocities | hits, text, tempo, z)

    At inference, z is sampled from N(0, I), so one hit pattern and prompt
    can produce multiple valid humanizations.
    """

    def __init__(self, text_emb_dim, d_model=192, latent_dim=32, n_heads=6, n_layers=3,
        dropout=0.1, n_steps=32, n_voices=9):
        super().__init__()

        self.n_steps = n_steps
        self.n_voices = n_voices
        self.n_tokens = n_steps * n_voices
        self.d_model = d_model
        self.latent_dim = latent_dim

        # Posterior encoder sees hit, target offset, and target velocity.
        self.performance_proj = nn.Linear(3, d_model)

        # Decoder sees only the fixed hit pattern.
        self.hit_proj = nn.Linear(1, d_model)

        self.step_emb = nn.Embedding(n_steps, d_model)
        self.voice_emb = nn.Embedding(n_voices, d_model)
        self.metric_emb = nn.Embedding(16, d_model)

        self.text_proj = TextConditionProjection(text_emb_dim=text_emb_dim, d_model=d_model)

        self.tempo_proj = nn.Sequential(
            nn.Linear(1, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
        )

        # Add the condition to every encoder token instead of relying on one
        # condition token that the Transformer can easily ignore.
        self.encoder_condition_fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        self.posterior_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.posterior_token = nn.Parameter(torch.zeros(1, 1, d_model))

        self.mu_head = nn.Linear(d_model, latent_dim)
        self.logvar_head = nn.Linear(d_model, latent_dim)

        self.latent_proj = nn.Sequential(
            nn.Linear(latent_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        # Each decoder token receives hit content, text/tempo condition, and z.
        self.decoder_condition_fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        # Non-autoregressive Transformer decoder where all 288 output
        # locations are predicted together because the hit score is fixed.
        self.performance_decoder = nn.TransformerEncoder(decoder_layer, num_layers=n_layers)

        self.offset_head = nn.Linear(d_model, 1)
        self.velocity_head = nn.Linear(d_model, 1)

        step_ids = []
        voice_ids = []
        metric_ids = []

        for step in range(n_steps):
            for voice in range(n_voices):
                step_ids.append(step)
                voice_ids.append(voice)
                metric_ids.append(step % 16)

        self.register_buffer(
            "step_ids",
            torch.tensor(step_ids, dtype=torch.long),
        )
        self.register_buffer(
            "voice_ids",
            torch.tensor(voice_ids, dtype=torch.long),
        )
        self.register_buffer(
            "metric_ids",
            torch.tensor(metric_ids, dtype=torch.long),
        )

        nn.init.normal_(self.posterior_token, std=0.02)

    def positional_tokens(self):
        return (
            self.step_emb(self.step_ids)[None, :, :]
            + self.voice_emb(self.voice_ids)[None, :, :]
            + self.metric_emb(self.metric_ids)[None, :, :]
        )

    def get_condition(self, prompt_emb, tempo):
        batch_size = prompt_emb.shape[0]

        text_condition = self.text_proj(prompt_emb)

        tempo_norm = (
            (tempo.view(batch_size, 1) - 120.0) / 40.0
        ).clamp(-3.0, 3.0)

        tempo_condition = self.tempo_proj(tempo_norm)

        return text_condition + tempo_condition

    def encode(
        self,
        hits,
        target_offsets,
        target_velocities,
        prompt_emb,
        tempo,
    ):
        batch_size = hits.shape[0]

        # Empty positions carry no target performance information.
        masked_offsets = target_offsets * hits
        masked_velocities = target_velocities * hits

        performance_features = torch.stack([hits, masked_offsets, masked_velocities],dim=-1).reshape(batch_size, self.n_tokens, 3)

        tokens = self.performance_proj(performance_features)
        tokens = tokens + self.positional_tokens()

        condition = self.get_condition(prompt_emb, tempo)
        condition_tokens = condition[:, None, :].expand(-1, self.n_tokens, -1)

        tokens = self.encoder_condition_fusion(torch.cat([tokens, condition_tokens], dim=-1))
        posterior_token = (self.posterior_token.expand(batch_size, -1, -1) + condition[:, None, :])
        encoded = self.posterior_encoder(torch.cat([posterior_token, tokens], dim=1))

        posterior_state = encoded[:, 0, :]

        mu = self.mu_head(posterior_state)
        logvar = self.logvar_head(posterior_state).clamp(-8.0, 8.0)

        return mu, logvar

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, hits, prompt_emb, tempo, z):
        batch_size = hits.shape[0]

        hit_features = hits.reshape(batch_size, self.n_tokens, 1)

        tokens = self.hit_proj(hit_features)
        tokens = tokens + self.positional_tokens()

        condition = self.get_condition(prompt_emb, tempo)
        latent_condition = self.latent_proj(z)

        condition_tokens = condition[:, None, :].expand(-1, self.n_tokens, -1)
        latent_tokens = latent_condition[:, None, :].expand(-1, self.n_tokens, -1)

        tokens = self.decoder_condition_fusion(
            torch.cat(
                [tokens, condition_tokens, latent_tokens],
                dim=-1,
            )
        )

        decoded = self.performance_decoder(tokens)

        offsets = 0.5 * torch.tanh(
            self.offset_head(decoded)
        ).reshape(batch_size, self.n_steps, self.n_voices)

        velocities = torch.sigmoid(
            self.velocity_head(decoded)
        ).reshape(batch_size, self.n_steps, self.n_voices)

        offsets = offsets * hits
        velocities = velocities * hits

        return offsets, velocities

    def forward(self, hits, target_offsets, target_velocities, prompt_emb, tempo, sample=True):
        mu, logvar = self.encode( hits=hits, target_offsets=target_offsets, target_velocities=target_velocities,
            prompt_emb=prompt_emb, tempo=tempo)

        z = (
            self.reparameterize(mu, logvar)
            if sample
            else mu
        )

        pred_offsets, pred_velocities = self.decode(hits=hits, prompt_emb=prompt_emb, tempo=tempo, z=z)

        return pred_offsets, pred_velocities, mu, logvar

In [374]:
# Initialize conditional VAE

LATENT_DIM = 16

model = ConditionalGrooveVAE(
    text_emb_dim=TEXT_EMB_DIM,
    d_model=192,
    latent_dim=LATENT_DIM,
    n_heads=6,
    n_layers=3,
    dropout=0.1,
    n_steps=32,
    n_voices=9,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4,
)

num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {num_params:,}")
print(f"Latent dimensions: {LATENT_DIM}")

Trainable parameters: 3,027,106
Latent dimensions: 16


In [375]:
# Inference cell

from pathlib import Path

from midi_data_preprocessing import (
    extract_patterns,
    reconstruct_midi,
    remap_midi_file,
)


checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()


@torch.no_grad()
def predict_with_prompt_vae(model, hit_matrix, tempo_value, prompt_text, z=None):
    hits = torch.as_tensor(
        hit_matrix,
        dtype=torch.float32,
        device=device,
    ).unsqueeze(0)
    tempo = torch.tensor([tempo_value], dtype=torch.float32, device=device)
    prompt_emb = encode_prompt_batch([prompt_text])

    if z is None:
        z = torch.randn(1, model.latent_dim, device=device)

    pred_offsets, pred_velocities = model.decode(
        hits=hits,
        prompt_emb=prompt_emb,
        tempo=tempo,
        z=z,
    )
    return pred_offsets[0].cpu().numpy(), pred_velocities[0].cpu().numpy()


input_path = Path(INPUT_MIDI_PATH).expanduser().resolve()
output_path = Path(OUTPUT_MIDI_PATH).expanduser().resolve()

if not input_path.is_file():
    raise FileNotFoundError(f"Input MIDI file not found: {input_path}")

input_midi = pretty_midi.PrettyMIDI(str(input_path))
_, detected_tempos = input_midi.get_tempo_changes()
tempo = float(TEMPO_OVERRIDE) if TEMPO_OVERRIDE is not None else float(detected_tempos[0])

mapped_midi = remap_midi_file(str(input_path))
patterns_H, _, _ = extract_patterns(mapped_midi, tempo)
if not patterns_H:
    raise ValueError("The input MIDI did not produce a two-bar drum pattern.")

hit_matrix = patterns_H[0]
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

predicted_offsets, predicted_velocities = predict_with_prompt_vae(
    model=model,
    hit_matrix=hit_matrix,
    tempo_value=tempo,
    prompt_text=PROMPT,
)

modified_midi = reconstruct_midi(
    hit_matrix,
    predicted_offsets,
    predicted_velocities,
    tempo,
    mean_offset=0,
)
output_path.parent.mkdir(parents=True, exist_ok=True)
modified_midi.write(str(output_path))

print(f"Prompt: {PROMPT}")
print(f"Tempo: {tempo:.2f} BPM")
print(f"Wrote modified MIDI to: {output_path}")


Prompt: Make this sound like jazz with swing, use gentle dynamics.
Tempo: 102.00 BPM
Wrote modified MIDI to: /Users/adrianlam/Documents/GitHub/Text2Groove/Project Code/modified_output.mid
